# Test Scopus API Key — Full Abstract Access

This notebook checks whether your **Elsevier / Scopus API key** works and whether it can
unlock **full abstracts** (not just truncated metadata).

### Notes on access
- A plain **API key** (no institutional token) generally works only from an **institutional IP** that is subscribed to Scopus.
- An **InstToken** (institutional token) lets the key work from off-campus / any IP.
- Full-length abstracts come from the **Abstract Retrieval API**. The Search API only returns short snippets.

Get a key here: https://dev.elsevier.com/

## 1. Configure credentials

In [1]:
import os
import requests
import json

# --- Put your key here, or set it as an environment variable ---
API_KEY = os.environ.get("SCOPUS_API_KEY", "57581543dea4d979a748eb7941383971")

# Optional: institutional token (InstToken). Leave as None if you don't have one.
INST_TOKEN = os.environ.get("SCOPUS_INST_TOKEN", None)

HEADERS = {
    "X-ELS-APIKey": API_KEY,
    "Accept": "application/json",
}
if INST_TOKEN:
    HEADERS["X-ELS-Insttoken"] = INST_TOKEN

assert API_KEY, "Set your API key first!"
print("Key configured. InstToken provided:", bool(INST_TOKEN))

Key configured. InstToken provided: False


## 2. Check what your key is entitled to

The `authenticate` / entitlement endpoints tell you which views and APIs your key can use.

In [2]:
# A lightweight probe: a tiny Scopus search. A 200 = key valid on this IP.
probe = requests.get(
    "https://api.elsevier.com/content/search/scopus",
    headers=HEADERS,
    params={"query": "TITLE(brain)", "count": 1},
)
print("Status:", probe.status_code)
print("Your IP seen by Elsevier:", probe.headers.get("X-ELS-APIKey", "(hidden)"))
if probe.status_code == 200:
    print("\n✅ API key is VALID from this network.")
elif probe.status_code in (401, 403):
    print("\n❌ Not authorized. Likely off an institutional IP and no InstToken,")
    print("   or the key lacks Scopus entitlement.")
    print(json.dumps(probe.json(), indent=2))
else:
    print(probe.text[:1000])

Status: 200
Your IP seen by Elsevier: 57581543dea4d979a748eb7941383971

✅ API key is VALID from this network.


## 3. Retrieve a FULL abstract (Abstract Retrieval API)

We request `view=FULL`. If your key is entitled, you get the complete abstract text.
If not, Elsevier silently downgrades to the basic view (short/no abstract).

In [3]:
# A well-known open-access DOI to test with. Swap for any DOI/Scopus ID you like.
TEST_DOI = "10.1016/j.cell.2011.02.013"  # Hanahan & Weinberg, Hallmarks of Cancer

def get_abstract(doi, view="FULL"):
    url = f"https://api.elsevier.com/content/abstract/doi/{doi}"
    r = requests.get(url, headers=HEADERS, params={"view": view})
    return r

r = get_abstract(TEST_DOI, view="FULL")
print("Status:", r.status_code)

if r.status_code == 200:
    data = r.json()
    core = data.get("abstracts-retrieval-response", {})
    coredata = core.get("coredata", {})
    title = coredata.get("dc:title")
    abstract = coredata.get("dc:description")
    print("\nTitle:", title)
    print("\nAbstract length:", len(abstract or ""), "chars")
    print("\n--- ABSTRACT ---\n")
    print(abstract or "(no abstract returned — key not entitled to FULL view)")
else:
    print(r.text[:1500])

Status: 200

Title: Hallmarks of cancer: The next generation

Abstract length: 1181 chars

--- ABSTRACT ---

The hallmarks of cancer comprise six biological capabilities acquired during the multistep development of human tumors. The hallmarks constitute an organizing principle for rationalizing the complexities of neoplastic disease. They include sustaining proliferative signaling, evading growth suppressors, resisting cell death, enabling replicative immortality, inducing angiogenesis, and activating invasion and metastasis. Underlying these hallmarks are genome instability, which generates the genetic diversity that expedites their acquisition, and inflammation, which fosters multiple hallmark functions. Conceptual progress in the last decade has added two emerging hallmarks of potential generality to this list - reprogramming of energy metabolism and evading immune destruction. In addition to cancer cells, tumors exhibit another dimension of complexity: they contain a repertoire of 

## 4. Compare FULL vs basic view

This is the real test: if `FULL` gives you noticeably more abstract text than the
default view, your key **unlocks full abstracts**.

In [4]:
def abstract_len(doi, view):
    r = get_abstract(doi, view=view)
    if r.status_code != 200:
        return None, r.status_code
    desc = (
        r.json()
        .get("abstracts-retrieval-response", {})
        .get("coredata", {})
        .get("dc:description")
    )
    return len(desc or ""), r.status_code

full_len, full_status = abstract_len(TEST_DOI, "FULL")
meta_len, meta_status = abstract_len(TEST_DOI, "META")

print(f"FULL view : status {full_status}, abstract {full_len} chars")
print(f"META view : status {meta_status}, abstract {meta_len} chars")

if full_len and full_len > 100:
    print("\n✅ Your key UNLOCKS full abstracts.")
elif full_status in (401, 403):
    print("\n❌ Not authorized for FULL abstracts (need institutional access / InstToken).")
else:
    print("\n⚠️ Abstract came back empty or truncated — limited entitlement.")

FULL view : status 200, abstract 1181 chars
META view : status 200, abstract 0 chars

✅ Your key UNLOCKS full abstracts.


## 5. (Optional) Quick interpretation

| Result | Meaning |
|---|---|
| 200 + long abstract | Full access works 🎉 |
| 200 + empty/short `dc:description` | Key valid but downgraded to basic view (no full-text entitlement) |
| 401 | API key missing/invalid |
| 403 | Valid key, but no entitlement from this IP — use an institutional network or an InstToken |
| 429 | Rate / quota limit hit — wait and retry |